# 04 — Attention Visualization & Streamlit App

**Pipeline:** Extract per-layer/head attention → Plotly heatmaps → generate `streamlit_app.py`

```
Any HuggingFace Transformer model
        │
        ▼
  Forward pass with output_attentions=True
        │  attention tensors: [batch, heads, seq, seq]
        ▼
  Per-head attention extraction
        │
        ├──── Plotly heatmap (inline notebook)
        └──── streamlit_app.py (interactive app)
                    │
                    ▼
             Deploy to Streamlit Cloud
             or HuggingFace Spaces
```

**References:**
- BertViz: Vig (2019) https://arxiv.org/abs/1906.05714
- Attention Patterns: Clark et al. (2019) https://arxiv.org/abs/1906.04341

> **CPU-friendly:** BERT-base-uncased runs on CPU; use a small input for fast iteration.

In [ ]:
from __future__ import annotations

import logging
import os
from pathlib import Path

import torch
import wandb

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CFG = {
    "model_name": "bert-base-uncased",
    "sample_text": "The Transformer architecture revolutionized natural language processing.",
    "layers_to_viz": [0, 6, 11],
    "device": DEVICE,
    "output_app": "streamlit_app.py",
}

if not os.getenv("WANDB_DISABLED"):
    wandb.init(project="ai-transformer-research-hub", config=CFG, job_type="attention-viz")


In [ ]:
# Load model, tokenize, run forward pass and extract attention weights
# BertViz: Vig (2019) https://arxiv.org/abs/1906.05714
# Attention Patterns: Clark et al. (2019) https://arxiv.org/abs/1906.04341


def extract_attention_weights(
    model_name: str,
    text: str,
    device: str,
) -> tuple[list[torch.Tensor], list[str]]:
    """Run a forward pass and return per-layer attention tensors + tokens.

    Uses output_attentions=True which is supported by all HuggingFace
    encoder and encoder-decoder models.

    Args:
        model_name: HuggingFace model identifier.
        text: Input sentence to visualise.
        device: Torch device string.

    Returns:
        Tuple of (attention_layers, tokens) where attention_layers is a list
        of tensors shaped [num_heads, seq_len, seq_len] and tokens is the
        list of decoded token strings.
    """
    try:
        from transformers import AutoModel, AutoTokenizer  # type: ignore
    except ImportError:
        log.warning("transformers not installed — returning synthetic attention weights")
        n_heads, n_tokens = 12, 12
        fake_attn = [
            torch.softmax(torch.randn(n_heads, n_tokens, n_tokens), dim=-1)
            for _ in range(12)
        ]
        return fake_attn, [f"[tok{i}]" for i in range(n_tokens)]

    log.info("Loading %s", model_name)
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name, output_attentions=True)
        model.eval().to(device)
    except Exception as exc:  # noqa: BLE001
        log.warning("Could not load %s: %s — using synthetic weights", model_name, exc)
        n_heads, n_tokens = 12, 10
        fake_attn = [
            torch.softmax(torch.randn(n_heads, n_tokens, n_tokens), dim=-1)
            for _ in range(12)
        ]
        return fake_attn, [f"[tok{i}]" for i in range(n_tokens)]

    inputs = tokenizer(text, return_tensors="pt").to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

    with torch.no_grad():
        outputs = model(**inputs)

    # Each element of outputs.attentions: [batch, heads, seq, seq] -> squeeze batch
    attention_layers = [attn[0].cpu() for attn in outputs.attentions]
    log.info(
        "Extracted attention: %d layers x %d heads x %d tokens",
        len(attention_layers),
        attention_layers[0].shape[0],
        attention_layers[0].shape[1],
    )
    return attention_layers, tokens


attention_layers, tokens = extract_attention_weights(
    CFG["model_name"], CFG["sample_text"], CFG["device"]
)


In [ ]:
import math
# Render Plotly attention heatmaps (one subplot per head, one figure per layer)
# BertViz: Vig (2019) https://arxiv.org/abs/1906.05714


def plot_attention_layer(
    attn: torch.Tensor,
    layer_idx: int,
    tokens: list[str],
):
    """Render a Plotly subplot grid with all attention heads for one layer.

    Args:
        attn: Tensor of shape [num_heads, seq_len, seq_len].
        layer_idx: Layer index for the figure title.
        tokens: Token strings for axis labels.

    Returns:
        Plotly Figure, or None if plotly is unavailable.
    """
    try:
        import plotly.graph_objects as go  # type: ignore
        from plotly.subplots import make_subplots  # type: ignore
    except ImportError:
        log.warning("plotly not installed — skipping visualisation")
        return None

    num_heads = attn.shape[0]
    cols = min(4, num_heads)
    rows = math.ceil(num_heads / cols)
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[f"Head {h}" for h in range(num_heads)],
    )
    for head in range(num_heads):
        row, col = head // cols + 1, head % cols + 1
        fig.add_trace(
            go.Heatmap(
                z=attn[head].numpy().tolist(),
                x=tokens, y=tokens,
                colorscale="Viridis",
                showscale=(head == 0),
            ),
            row=row, col=col,
        )
    fig.update_layout(title_text=f"Layer {layer_idx} — Attention Heads", height=300 * rows)
    fig.show()
    return fig


figures: list = []
for layer_idx in CFG["layers_to_viz"]:
    if layer_idx < len(attention_layers):
        fig = plot_attention_layer(attention_layers[layer_idx], layer_idx, tokens)
        if fig is not None:
            figures.append((layer_idx, fig))

if not os.getenv("WANDB_DISABLED") and figures:
    wandb.log({f"attention_layer_{li}": wandb.Html(f.to_html()) for li, f in figures})


In [ ]:
# Generate the deployable Streamlit attention heatmap app
# BertViz: Vig (2019) https://arxiv.org/abs/1906.05714

_STREAMLIT_APP_LINES = [
    "#!/usr/bin/env python",
    "# streamlit_app.py -- Interactive Attention Token Viz",
    "# Generated by notebooks/04_attention_viz_streamlit.ipynb",
    "# Run with:  streamlit run streamlit_app.py",
    "# References:",
    "#   BertViz: Vig (2019) https://arxiv.org/abs/1906.05714",
    "#   Attention Patterns: Clark et al. (2019) https://arxiv.org/abs/1906.04341",
    "from __future__ import annotations",
    "import math",
    "import os",
    "import streamlit as st",
    "import torch",
    "os.environ.setdefault('WANDB_DISABLED', 'true')",
    "st.set_page_config(page_title='Attention Token Viz', layout='wide')",
    "st.title('Attention Token Viz')",
    "st.caption('BertViz-style heatmaps -- Vig (2019) https://arxiv.org/abs/1906.05714')",
    "with st.sidebar:",
    "    st.header('Settings')",
    "    model_name = st.text_input('HuggingFace model ID', value='bert-base-uncased')",
    "    input_text = st.text_area(",
    "        'Input text',",
    "        value='The Transformer architecture revolutionized natural language processing.',",
    "    )",
    "    run_btn = st.button('Run')",
    "if run_btn or 'attention_layers' not in st.session_state:",
    "    with st.spinner(f'Loading {model_name}...'):",
    "        try:",
    "            from transformers import AutoModel, AutoTokenizer",
    "            tokenizer = AutoTokenizer.from_pretrained(model_name)",
    "            model = AutoModel.from_pretrained(model_name, output_attentions=True)",
    "            model.eval()",
    "            inputs = tokenizer(input_text, return_tensors='pt')",
    "            tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())",
    "            with torch.no_grad():",
    "                outputs = model(**inputs)",
    "            st.session_state['attention_layers'] = [a[0].cpu() for a in outputs.attentions]",
    "            st.session_state['tokens'] = tokens",
    "            st.success(f'Loaded {model_name}')",
    "        except Exception as exc:",
    "            st.error(f'Could not load model: {exc}')",
    "            st.stop()",
    "if 'attention_layers' in st.session_state:",
    "    layers = st.session_state['attention_layers']",
    "    tokens = st.session_state['tokens']",
    "    n_layers = len(layers)",
    "    layer_idx = st.slider('Layer', 0, n_layers - 1, n_layers // 2)",
    "    attn = layers[layer_idx]",
    "    num_heads = attn.shape[0]",
    "    head_idx = st.slider('Head', 0, num_heads - 1, 0)",
    "    try:",
    "        import plotly.graph_objects as go",
    "        fig = go.Figure(go.Heatmap(",
    "            z=attn[head_idx].numpy().tolist(),",
    "            x=tokens, y=tokens,",
    "            colorscale='Viridis',",
    "            colorbar=dict(title='Attention weight'),",
    "        ))",
    "        fig.update_layout(",
    "            title=f'Layer {layer_idx} -- Head {head_idx}',",
    "            xaxis_title='Key tokens', yaxis_title='Query tokens', height=600,",
    "        )",
    "        st.plotly_chart(fig, use_container_width=True)",
    "    except ImportError:",
    "        st.warning('Install plotly: pip install plotly')",
    "        st.write(attn[head_idx].numpy())",
]

STREAMLIT_APP = "\n".join(_STREAMLIT_APP_LINES)

app_path = Path(CFG["output_app"])
app_path.write_text(STREAMLIT_APP + "\n")
log.info("Streamlit app written to %s", app_path)

if not os.getenv("WANDB_DISABLED"):
    wandb.log({"streamlit_app_generated": True})
    wandb.finish()
